# Merged Data v3
**Merge plan:**
- **anadata** → aggregated to one row per episode (`RF_EPISODE2`): diagnosis codes joined with ` | `
- **recete_enriched** → aggregated to one row per episode (`RF_EPISODE`): drug info joined with ` | `
- Recete joined onto anadata on episode ID (left join)
- **lab** → windowed join: most recent value per test within `LAB_WINDOW` days before each episode
- Empty strings (`""`, `"-"`, `"--"`, `"yok"`, etc.) → `np.nan` throughout

In [12]:
# ── Step 1: Config & Load ─────────────────────────────────────────────────────
import pyarrow.parquet as pq
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
from dateutil.relativedelta import relativedelta

pd.set_option('future.no_silent_downcasting', True)

PARQUET_DIR = Path('/home/alperbulbul/ACU/kutsii_workspace/parquet')
CUTOFF      = datetime.now() - relativedelta(years=1)
LAB_WINDOW  = pd.Timedelta('7 days')
SAMPLE_N    = 10_000   # set to None to run on full dataset

EMPTY_VALS = ['', '-', '--', '---', 'yok', 'Yok', 'YOK', 'N/A', 'n/a', 'NA', 'nan']

def clean_object_cols(df):
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].str.strip().replace(EMPTY_VALS, np.nan)
    return df

def join_unique(s):
    vals = s.dropna().unique()
    return ' | '.join(str(v) for v in vals) if len(vals) > 0 else np.nan

# anadata
anadata = pq.read_table(str(PARQUET_DIR / 'anadata.parquet')).to_pandas()
anadata['EPISODE_TARIH'] = pd.to_datetime(anadata['EPISODE_TARIH'], errors='coerce')
anadata = anadata[anadata['EPISODE_TARIH'] >= CUTOFF].dropna(subset=['HASTA_ID']).copy()
anadata = clean_object_cols(anadata)
print(f'anadata : {len(anadata):,} rows | {anadata["RF_EPISODE2"].nunique():,} unique episodes')

# lab
lab = pq.read_table(str(PARQUET_DIR / 'lab.parquet')).to_pandas()
lab['REP_DATE'] = pd.to_datetime(lab['REP_DATE'], errors='coerce')
lab = lab[lab['REP_DATE'] >= CUTOFF].copy()
print(f'lab     : {len(lab):,} rows')

# recete_enriched
recete = pq.read_table(str(PARQUET_DIR / 'recete_enriched.parquet')).to_pandas()
recete['RECETE_TARIH'] = pd.to_datetime(recete['RECETE_TARIH'], errors='coerce')
recete = recete[recete['RECETE_TARIH'] >= CUTOFF].copy()
recete = clean_object_cols(recete)
print(f'recete  : {len(recete):,} rows')

anadata : 162,879 rows | 48,466 unique episodes
lab     : 5,502,086 rows
recete  : 587,474 rows


In [13]:
# ── Step 2: Aggregate anadata → one row per episode ──────────────────────────
# Diagnosis columns vary across rows (one row per diagnosis); everything else is constant.
DIAG_COLS = ['TANIKODU', 'TUM_EPS_TANILAR', 'TANITARIH']
FIRST_COLS = [c for c in anadata.columns if c not in DIAG_COLS + ['RF_EPISODE2']]

agg_spec = {col: (col, join_unique) for col in DIAG_COLS if col in anadata.columns}
agg_spec.update({col: (col, 'first') for col in FIRST_COLS})

anadata_agg = (
    anadata.groupby('RF_EPISODE2')
    .agg(**agg_spec)
    .reset_index()
)
anadata_agg = clean_object_cols(anadata_agg)

print(f'anadata_agg : {len(anadata_agg):,} rows x {anadata_agg.shape[1]} cols')
print(f'Unique diagnoses per episode (sample):')
print(anadata_agg['TANIKODU'].str.count(r'\|').add(1).describe().round(1))

anadata_agg : 48,466 rows x 61 cols
Unique diagnoses per episode (sample):
count    48466.0
mean         1.6
std          1.1
min          1.0
25%          1.0
50%          1.0
75%          2.0
max         16.0
Name: TANIKODU, dtype: float64


In [14]:
# ── Step 3: Aggregate recete → one row per episode ───────────────────────────
recete_agg = (
    recete.groupby('RF_EPISODE').agg(
        HASTA_ID         = ('HASTA_ID',        'first'),
        RECETE_TARIH     = ('RECETE_TARIH',     'first'),
        ILAC_ADI         = ('İlaç Adı',         join_unique),
        SIKLIK_DOZ       = ('Sıklık X Doz',     join_unique),
        VERILID_YOLU     = ('VERİLİS_YOLU',     join_unique),
        GUN              = ('Gün',              join_unique),
        ILAC_KATEGORI    = ('ILAC_KATEGORI',    join_unique),
        ILAC_ICD10_KOD   = ('ILAC_ICD10_KOD',  join_unique),
        HASTALIK_SIDDETI = ('HASTALIK_SIDDETI', join_unique),
        ILAC_FIYAT       = ('ILAC_FIYAT',       lambda x: x.sum(min_count=1)),
    )
    .reset_index()
)
recete_agg = recete_agg.rename(
    columns={c: f'REC_{c}' for c in recete_agg.columns if c != 'RF_EPISODE'}
)
print(f'recete_agg: {len(recete_agg):,} rows | {recete_agg.shape[1]} cols')

recete_agg: 180,058 rows | 11 cols


In [15]:
# ── Step 4: Join recete onto anadata_agg on episode ID ───────────────────────
anadata_agg['RF_EPISODE2']  = anadata_agg['RF_EPISODE2'].astype(str)
recete_agg['RF_EPISODE']    = recete_agg['RF_EPISODE'].astype(str)

merged = anadata_agg.merge(
    recete_agg.drop(columns=['REC_HASTA_ID'], errors='ignore'),
    left_on='RF_EPISODE2',
    right_on='RF_EPISODE',
    how='left',
)
merged = merged.drop(columns=['RF_EPISODE'], errors='ignore')

matched = merged['REC_RECETE_TARIH'].notna().sum()
print(f'After recete join : {merged.shape[0]:,} rows x {merged.shape[1]} cols')
print(f'Episodes with Rx  : {matched:,} ({matched / len(merged) * 100:.1f}%)')

# Sanity check: ID overlap
rec_ids = set(recete_agg['RF_EPISODE'])
ana_ids = set(anadata_agg['RF_EPISODE2'])
print(f'Recete IDs matched in anadata : {len(rec_ids & ana_ids):,} / {len(rec_ids):,}')

After recete join : 48,466 rows x 70 cols
Episodes with Rx  : 14,209 (29.3%)
Recete IDs matched in anadata : 14,209 / 180,058


In [16]:
# How many recete IDs exist in anadata?
rec_ids = set(recete_agg['RF_EPISODE'])
ana_ids = set(anadata['RF_EPISODE2'])
print(f"Recete IDs in anadata  : {len(rec_ids & ana_ids):,}")
print(f"Recete IDs NOT in anadata: {len(rec_ids - ana_ids):,}")
print(f"'nan' in recete IDs?   : {'nan' in rec_ids}")
print(f"'nan' in anadata IDs?  : {'nan' in ana_ids}")


Recete IDs in anadata  : 14,209
Recete IDs NOT in anadata: 165,849
'nan' in recete IDs?   : False
'nan' in anadata IDs?  : False


In [17]:
# ── Step 5: Windowed lab join (all test types) ────────────────────────────────
# Optionally sample episodes for faster iteration; set SAMPLE_N=None for full run
base = merged.sample(n=SAMPLE_N, random_state=42).copy() if SAMPLE_N else merged.copy()
print(f'Working on {len(base):,} episodes')

# Filter lab to only the sampled patients — keeps memory manageable
sample_patients = set(base['HASTA_ID'].unique())
lab_f = lab[lab['HASTA_ID'].isin(sample_patients)].dropna(subset=['SUB_CODE', 'RESULT']).copy()
print(f'lab after patient filter: {len(lab_f):,} rows | {lab_f["SUB_CODE"].nunique():,} test types')

# Join on patient, then filter to [episode - LAB_WINDOW, episode]
lab_ep = base[['HASTA_ID', 'EPISODE_TARIH']].merge(
    lab_f[['HASTA_ID', 'REP_DATE', 'SUB_CODE', 'RESULT']],
    on='HASTA_ID', how='left'
)
lab_ep = lab_ep[
    (lab_ep['REP_DATE'] <= lab_ep['EPISODE_TARIH']) &
    (lab_ep['REP_DATE'] >= lab_ep['EPISODE_TARIH'] - LAB_WINDOW)
]

# Most recent value per (episode, test) → pivot wide
lab_wide = (
    lab_ep.sort_values('REP_DATE')
    .groupby(['HASTA_ID', 'EPISODE_TARIH', 'SUB_CODE'])['RESULT']
    .last()
    .unstack('SUB_CODE')
    .reset_index()
)
lab_wide.columns = (
    ['HASTA_ID', 'EPISODE_TARIH']
    + [f'LAB_{c}' for c in lab_wide.columns[2:]]
)

base = base.merge(lab_wide, on=['HASTA_ID', 'EPISODE_TARIH'], how='left')

matched_lab = base[[c for c in base.columns if c.startswith('LAB_')]].notna().any(axis=1).sum()
print(f'After lab join    : {base.shape[0]:,} rows x {base.shape[1]} cols')
print(f'Episodes with any lab ({LAB_WINDOW.days}d window): {matched_lab:,} ({matched_lab/len(base)*100:.1f}%)')

Working on 10,000 episodes
lab after patient filter: 431,292 rows | 1,544 test types
After lab join    : 10,000 rows x 851 cols
Episodes with any lab (7d window): 2,379 (23.8%)


In [18]:
# ── Visit frequency distribution ──────────────────────────────────────────────
visit_counts = merged_sample.groupby('HASTA_ID').size()

freq_dist = (
    visit_counts.value_counts()
    .sort_index()
    .reset_index()
)
freq_dist.columns = ['visits', 'num_patients']
freq_dist['num_episodes'] = freq_dist['visits'] * freq_dist['num_patients']

print(f"{'Visits':>8} {'Patients':>10} {'Episodes':>10}")
print("-" * 30)
for _, row in freq_dist.iterrows():
    print(f"{int(row['visits']):>8} {int(row['num_patients']):>10,} {int(row['num_episodes']):>10,}")
print("-" * 30)
print(f"{'TOTAL':>8} {visit_counts.nunique():>10,} {len(merged_sample):>10,}")
print(f"\nMedian visits per patient : {visit_counts.median():.0f}")
print(f"Mean   visits per patient : {visit_counts.mean():.1f}")
print(f"Max    visits per patient : {visit_counts.max()}")


  Visits   Patients   Episodes
------------------------------
       1      2,851      2,851
       2        956      1,912
       3        421      1,263
       4        194        776
       5         96        480
       6         71        426
       7         39        273
       8         24        192
       9         17        153
      10          7         70
      11         12        132
      12          5         60
      13         11        143
      14          4         56
      15          3         45
      17          1         17
      18          4         72
      20          1         20
      21          1         21
      22          2         44
      24          3         72
      29          1         29
------------------------------
   TOTAL         22      9,107

Median visits per patient : 1
Mean   visits per patient : 1.9
Max    visits per patient : 29


In [19]:
# ── Step 6: Post-processing ───────────────────────────────────────────────────
base = clean_object_cols(base)

before = len(base)
base = base.drop_duplicates()
after = len(base)

print(f'Rows before dedup : {before:,}')
print(f'Rows after dedup  : {after:,}')
print(f'Duplicates removed: {before - after:,}')
print(f'Unique patients   : {base["HASTA_ID"].nunique():,}')
print(f'Final shape       : {base.shape}')

# Visit frequency distribution
visit_counts = base.groupby('HASTA_ID').size()
freq_dist = visit_counts.value_counts().sort_index().reset_index()
freq_dist.columns = ['visits', 'num_patients']
freq_dist['num_episodes'] = freq_dist['visits'] * freq_dist['num_patients']

print(f"\n{'Visits':>8} {'Patients':>10} {'Episodes':>10}")
print("-" * 30)
for _, row in freq_dist.iterrows():
    print(f"{int(row['visits']):>8} {int(row['num_patients']):>10,} {int(row['num_episodes']):>10,}")
print("-" * 30)
print(f"Median visits/patient : {visit_counts.median():.0f}")
print(f"Mean   visits/patient : {visit_counts.mean():.1f}")
print(f"Max    visits/patient : {visit_counts.max()}")

Rows before dedup : 10,000
Rows after dedup  : 10,000
Duplicates removed: 0
Unique patients   : 5,875
Final shape       : (10000, 851)

  Visits   Patients   Episodes
------------------------------
       1      3,758      3,758
       2      1,259      2,518
       3        489      1,467
       4        160        640
       5         77        385
       6         34        204
       7         26        182
       8         23        184
       9         10         90
      10          8         80
      11          6         66
      12          6         72
      13          4         52
      14          1         14
      15          2         30
      16          3         48
      18          3         54
      19          1         19
      20          1         20
      22          1         22
      25          1         25
      29          1         29
      41          1         41
------------------------------
Median visits/patient : 1
Mean   visits/patient : 1.7
Max 

In [20]:
print(anadata['RF_EPISODE2'].value_counts().head(10))
print(f"Unique RF_EPISODE2 : {anadata['RF_EPISODE2'].nunique():,} out of {len(anadata):,} rows")

RF_EPISODE2
3574342     916
2943166     556
43482328    480
3064977     438
2956531     434
3677466     304
3109833     265
3610323     254
2956772     227
35027309    225
Name: count, dtype: int64
Unique RF_EPISODE2 : 48,466 out of 162,879 rows


In [21]:
# ── Diagnostic: check RF_EPISODE2 uniqueness ──────────────────────────────────
most_repeated = anadata[anadata['RF_EPISODE2'] == anadata['RF_EPISODE2'].value_counts().index[0]]
varying = most_repeated.nunique()
print("Columns that vary (nunique > 1):")
print(varying[varying > 1])
print("\nColumns with all NaN:")
print(varying[varying == 0].index.tolist())
print(f"\nUnique RF_EPISODE2 : {anadata['RF_EPISODE2'].nunique():,} out of {len(anadata):,} rows")


Columns that vary (nunique > 1):
EPISODE_TARIH      2
TANITARIH          2
TANIKODU           2
TUM_EPS_TANILAR    2
dtype: int64

Columns with all NaN:
['Düşme Riski', 'Ağrısı var mı', 'Ağrı skoru', 'Sıklık', 'Muayene Notu', 'Tedavi Notu', 'Özgeçmiş Notu', 'Sigara', 'Alkol', 'Alerji', 'Yer', 'Nitelik', 'Madde', 'Kontrol Notu', 'ilaç Alerjisi', 'Hipertansiyon Hastada', 'Kalp Damar Hastada', 'Diyabet Hastada', 'Kan Hastalıkları Hastada', 'Kronik Hastalıklar Diğer', 'Ameliyat Geçmişi', 'Yaralanma Geçmişi', 'Sürekli Kullandığı İlaçlar', 'Kan Grubu', 'Engellilik', 'Anne KH', 'Baba KH', 'Erkek Kardeş KH', 'Kız Kardeş KH', 'Soygeçmiş Notu', 'Meslek', 'Sosyal Durum', 'Boy', 'Kilo', 'BMI', 'SPO2', 'Nabız', 'Ritmik/ Aritmik', 'KB-S', 'KB-D']

Unique RF_EPISODE2 : 48,466 out of 162,879 rows


In [22]:
# Check null % across full anadata
null_pct = anadata[['Düşme Riski', 'Ağrısı var mı', 'Ağrı skoru', 'Boy', 'Kilo', 'BMI',
                     'SPO2', 'Nabız', 'KB-S', 'KB-D', 'Sigara', 'Alkol']].isnull().mean().sort_values(ascending=False)
print(null_pct.round(3))


SPO2             0.997
BMI              0.996
Nabız            0.966
Ağrı skoru       0.883
Boy              0.801
KB-D             0.793
KB-S             0.793
Kilo             0.787
Alkol            0.165
Sigara           0.152
Düşme Riski      0.119
Ağrısı var mı    0.105
dtype: float64


In [23]:
all_nan_cols = ['Düşme Riski', 'Ağrısı var mı', 'Ağrı skoru', 'Sıklık', 'Muayene Notu',
                'Tedavi Notu', 'Özgeçmiş Notu', 'Sigara', 'Alkol', 'Alerji', 'Yer',
                'Nitelik', 'Madde', 'Kontrol Notu', 'ilaç Alerjisi', 'Hipertansiyon Hastada',
                'Kalp Damar Hastada', 'Diyabet Hastada', 'Kan Hastalıkları Hastada',
                'Kronik Hastalıklar Diğer', 'Ameliyat Geçmişi', 'Yaralanma Geçmişi',
                'Sürekli Kullandığı İlaçlar', 'Kan Grubu', 'Engellilik', 'Anne KH',
                'Baba KH', 'Erkek Kardeş KH', 'Kız Kardeş KH', 'Soygeçmiş Notu',
                'Meslek', 'Sosyal Durum', 'Boy', 'Kilo', 'BMI', 'SPO2', 'Nabız',
                'Ritmik/ Aritmik', 'KB-S', 'KB-D']

null_pct = anadata[all_nan_cols].isnull().mean().sort_values()
print("Columns to KEEP (< 50% null):")
print(null_pct[null_pct < 0.5].round(3))
print("\nColumns to DROP (>= 50% null):")
print(null_pct[null_pct >= 0.5].round(3))


Columns to KEEP (< 50% null):
Tedavi Notu         0.081
Muayene Notu        0.082
Ağrısı var mı       0.105
Düşme Riski         0.119
Ameliyat Geçmişi    0.152
Sigara              0.152
Alkol               0.165
Kan Grubu           0.171
Madde               0.387
dtype: float64

Columns to DROP (>= 50% null):
Yaralanma Geçmişi             0.562
Meslek                        0.570
Kontrol Notu                  0.579
Özgeçmiş Notu                 0.592
Sürekli Kullandığı İlaçlar    0.604
Engellilik                    0.617
Soygeçmiş Notu                0.633
Baba KH                       0.654
Anne KH                       0.674
Alerji                        0.751
Sosyal Durum                  0.757
Kilo                          0.787
KB-S                          0.793
KB-D                          0.793
Boy                           0.801
Erkek Kardeş KH               0.823
Kız Kardeş KH                 0.827
Ağrı skoru                    0.883
Kronik Hastalıklar Diğer      0.891
Hiper

In [24]:
# ── Step 7: Save ─────────────────────────────────────────────────────────────
suffix = f'_sample{SAMPLE_N}' if SAMPLE_N else ''
out_path = Path(f'/home/alperbulbul/ACU/kutsii_workspace/merged_new_v3{suffix}.csv')
base.to_csv(out_path, index=False)
print(f'Saved : {out_path}')
print(f'Shape : {base.shape[0]:,} rows | {base.shape[1]} columns')

Saved : /home/alperbulbul/ACU/kutsii_workspace/merged_new_v3_sample10000.csv
Shape : 10,000 rows | 851 columns


# 3-Year Dataset — Patients with > 2 Visits
**Changes from v3 (1-year):**
- `CUTOFF` extended to 3 years
- Final dataset filtered to patients with **more than 2 visits** (i.e. ≥ 3 episodes)
- No random sampling (`SAMPLE_N = None`)


In [25]:
# ── 3Y Step 1: Config & Load ───────────────────────────────────────────────────
CUTOFF_3Y = datetime.now() - relativedelta(years=3)
MIN_VISITS = 2  # strictly greater than this

# anadata
anadata_3y = pq.read_table(str(PARQUET_DIR / 'anadata.parquet')).to_pandas()
anadata_3y['EPISODE_TARIH'] = pd.to_datetime(anadata_3y['EPISODE_TARIH'], errors='coerce')
anadata_3y = anadata_3y[anadata_3y['EPISODE_TARIH'] >= CUTOFF_3Y].dropna(subset=['HASTA_ID']).copy()
anadata_3y = clean_object_cols(anadata_3y)
print(f'anadata_3y : {len(anadata_3y):,} rows | {anadata_3y["RF_EPISODE2"].nunique():,} unique episodes')

# lab
lab_3y = pq.read_table(str(PARQUET_DIR / 'lab.parquet')).to_pandas()
lab_3y['REP_DATE'] = pd.to_datetime(lab_3y['REP_DATE'], errors='coerce')
lab_3y = lab_3y[lab_3y['REP_DATE'] >= CUTOFF_3Y].copy()
print(f'lab_3y     : {len(lab_3y):,} rows')

# recete
recete_3y = pq.read_table(str(PARQUET_DIR / 'recete_enriched.parquet')).to_pandas()
recete_3y['RECETE_TARIH'] = pd.to_datetime(recete_3y['RECETE_TARIH'], errors='coerce')
recete_3y = recete_3y[recete_3y['RECETE_TARIH'] >= CUTOFF_3Y].copy()
recete_3y = clean_object_cols(recete_3y)
print(f'recete_3y  : {len(recete_3y):,} rows')

anadata_3y : 517,013 rows | 148,412 unique episodes


KeyboardInterrupt: 

In [ ]:
# ── 3Y Step 2: Aggregate anadata → one row per episode ────────────────────────
DIAG_COLS_3Y = ['TANIKODU', 'TUM_EPS_TANILAR', 'TANITARIH']
FIRST_COLS_3Y = [c for c in anadata_3y.columns if c not in DIAG_COLS_3Y + ['RF_EPISODE2']]

agg_spec_3y = {col: (col, join_unique) for col in DIAG_COLS_3Y if col in anadata_3y.columns}
agg_spec_3y.update({col: (col, 'first') for col in FIRST_COLS_3Y})

anadata_3y_agg = (
    anadata_3y.groupby('RF_EPISODE2')
    .agg(**agg_spec_3y)
    .reset_index()
)
anadata_3y_agg = clean_object_cols(anadata_3y_agg)
print(f'anadata_3y_agg : {len(anadata_3y_agg):,} rows x {anadata_3y_agg.shape[1]} cols')

In [ ]:
# ── 3Y Step 3: Aggregate recete → one row per episode ─────────────────────────
recete_3y_agg = (
    recete_3y.groupby('RF_EPISODE').agg(
        HASTA_ID         = ('HASTA_ID',        'first'),
        RECETE_TARIH     = ('RECETE_TARIH',     'first'),
        ILAC_ADI         = ('İlaç Adı',         join_unique),
        SIKLIK_DOZ       = ('Sıklık X Doz',     join_unique),
        VERILID_YOLU     = ('VERİLİS_YOLU',     join_unique),
        GUN              = ('Gün',              join_unique),
        ILAC_KATEGORI    = ('ILAC_KATEGORI',    join_unique),
        ILAC_ICD10_KOD   = ('ILAC_ICD10_KOD',  join_unique),
        HASTALIK_SIDDETI = ('HASTALIK_SIDDETI', join_unique),
        ILAC_FIYAT       = ('ILAC_FIYAT',       lambda x: x.sum(min_count=1)),
    )
    .reset_index()
)
recete_3y_agg = recete_3y_agg.rename(
    columns={c: f'REC_{c}' for c in recete_3y_agg.columns if c != 'RF_EPISODE'}
)
print(f'recete_3y_agg : {len(recete_3y_agg):,} rows | {recete_3y_agg.shape[1]} cols')

In [ ]:
# ── 3Y Step 4: Join recete onto anadata_3y_agg ────────────────────────────────
anadata_3y_agg['RF_EPISODE2'] = anadata_3y_agg['RF_EPISODE2'].astype(str)
recete_3y_agg['RF_EPISODE']   = recete_3y_agg['RF_EPISODE'].astype(str)

merged_3y = anadata_3y_agg.merge(
    recete_3y_agg.drop(columns=['REC_HASTA_ID'], errors='ignore'),
    left_on='RF_EPISODE2',
    right_on='RF_EPISODE',
    how='left',
)
merged_3y = merged_3y.drop(columns=['RF_EPISODE'], errors='ignore')

matched = merged_3y['REC_RECETE_TARIH'].notna().sum()
print(f'After recete join : {merged_3y.shape[0]:,} rows x {merged_3y.shape[1]} cols')
print(f'Episodes with Rx  : {matched:,} ({matched / len(merged_3y) * 100:.1f}%)')

In [ ]:
# ── 3Y Step 5: Windowed lab join ───────────────────────────────────────────────
sample_patients_3y = set(merged_3y['HASTA_ID'].unique())
lab_3y_f = lab_3y[lab_3y['HASTA_ID'].isin(sample_patients_3y)].dropna(subset=['SUB_CODE', 'RESULT']).copy()
print(f'lab_3y after patient filter: {len(lab_3y_f):,} rows | {lab_3y_f["SUB_CODE"].nunique():,} test types')

lab_ep_3y = merged_3y[['HASTA_ID', 'EPISODE_TARIH']].merge(
    lab_3y_f[['HASTA_ID', 'REP_DATE', 'SUB_CODE', 'RESULT']],
    on='HASTA_ID', how='left'
)
lab_ep_3y = lab_ep_3y[
    (lab_ep_3y['REP_DATE'] <= lab_ep_3y['EPISODE_TARIH']) &
    (lab_ep_3y['REP_DATE'] >= lab_ep_3y['EPISODE_TARIH'] - LAB_WINDOW)
]

lab_wide_3y = (
    lab_ep_3y.sort_values('REP_DATE')
    .groupby(['HASTA_ID', 'EPISODE_TARIH', 'SUB_CODE'])['RESULT']
    .last()
    .unstack('SUB_CODE')
    .reset_index()
)
lab_wide_3y.columns = (
    ['HASTA_ID', 'EPISODE_TARIH']
    + [f'LAB_{c}' for c in lab_wide_3y.columns[2:]]
)

merged_3y = merged_3y.merge(lab_wide_3y, on=['HASTA_ID', 'EPISODE_TARIH'], how='left')

matched_lab = merged_3y[[c for c in merged_3y.columns if c.startswith('LAB_')]].notna().any(axis=1).sum()
print(f'After lab join : {merged_3y.shape[0]:,} rows x {merged_3y.shape[1]} cols')
print(f'Episodes with any lab ({LAB_WINDOW.days}d window): {matched_lab:,} ({matched_lab/len(merged_3y)*100:.1f}%)')

In [ ]:
# ── 3Y Step 6: Filter to patients with > 2 visits ─────────────────────────────
merged_3y = clean_object_cols(merged_3y)
merged_3y = merged_3y.drop_duplicates()

visit_counts_3y = merged_3y.groupby('HASTA_ID')['RF_EPISODE2'].nunique()
multi_visit_ids = visit_counts_3y[visit_counts_3y > MIN_VISITS].index

before = len(merged_3y)
merged_3y = merged_3y[merged_3y['HASTA_ID'].isin(multi_visit_ids)].copy()
after = len(merged_3y)

print(f'Patients removed (≤ {MIN_VISITS} visits) : {len(visit_counts_3y) - len(multi_visit_ids):,}')
print(f'Patients kept    (> {MIN_VISITS} visits) : {len(multi_visit_ids):,}')
print(f'Episodes before filter : {before:,}')
print(f'Episodes after  filter : {after:,}')
print(f'Final shape            : {merged_3y.shape}')

# Visit frequency distribution
vc = merged_3y.groupby('HASTA_ID')['RF_EPISODE2'].nunique()
print(f'\nMedian visits/patient : {vc.median():.0f}')
print(f'Mean   visits/patient : {vc.mean():.1f}')
print(f'Max    visits/patient : {vc.max()}')

In [ ]:
# ── 3Y Step 7: Save ────────────────────────────────────────────────────────────
out_path_3y = Path('/home/alperbulbul/ACU/kutsii_workspace/merged_3y_min3visits.parquet')
merged_3y.to_parquet(out_path_3y, index=False)
print(f'Saved : {out_path_3y}')
print(f'Shape : {merged_3y.shape[0]:,} rows | {merged_3y.shape[1]} columns')

# ICD-10 Severity Enrichment — merged_new_v3_sample10000
Match `TANIKODU` against `/home/alperbulbul/ACU/icd10_diseases.json` and add `SEVERITY_LEVEL`.
- Supports pipe-separated codes in `TANIKODU` (takes the **max** severity across all codes)
- Lookup order: exact match → strip subcategory (e.g. `L30.9 → L30`) → range match (e.g. `A00-A09`)

In [2]:
import numpy as np
import json, re

# ── Load ICD-10 severity mapping ───────────────────────────────────────────────
with open('/home/alperbulbul/ACU/icd10_diseases.json') as f:
    icd10_data = json.load(f)

# Build lookups
exact_lookup = {}   # code -> severity_level
range_lookup = []   # list of (letter, lo_num, hi_num, severity_level)

for entry in icd10_data:
    code = entry['TANIKODU'].strip()
    sev  = entry['SEVERITY_LEVEL']
    if '-' in code:
        # e.g. "A00-A09" or "A92-A99"
        m = re.match(r'^([A-Z])(\d+)-[A-Z]?(\d+)$', code)
        if m:
            range_lookup.append((m.group(1), int(m.group(2)), int(m.group(3)), sev))
    else:
        exact_lookup[code] = sev

print(f'Exact codes loaded : {len(exact_lookup):,}')
print(f'Range codes loaded : {len(range_lookup):,}')

def lookup_severity(code):
    code = code.strip()
    # 1. Exact match
    if code in exact_lookup:
        return exact_lookup[code]
    # 2. Strip subcategory (e.g. L30.9 → L30)
    base = code.split('.')[0]
    if base in exact_lookup:
        return exact_lookup[base]
    # 3. Range match (letter + numeric part)
    m = re.match(r'^([A-Z])(\d+)', base)
    if m:
        letter, num = m.group(1), int(m.group(2))
        for r_letter, lo, hi, sev in range_lookup:
            if letter == r_letter and lo <= num <= hi:
                return sev
    return np.nan

# Quick sanity check
for test_code in ['L30.9', 'A09', 'J20.9', 'K30', 'Z00.0', 'UNKNOWN']:
    print(f'  {test_code:12s} → severity {lookup_severity(test_code)}')

Exact codes loaded : 9,851
Range codes loaded : 274
  L30.9        → severity 1
  A09          → severity 4
  J20.9        → severity 4
  K30          → severity 3
  Z00.0        → severity 1
  UNKNOWN      → severity nan


In [5]:
import pandas as pd
from pathlib import Path
# ── Load the sample CSV ────────────────────────────────────────────────────────
csv_path = Path('/home/alperbulbul/ACU/kutsii_workspace/merged_new_v3_sample10000.csv')
df_sample = pd.read_csv(csv_path, low_memory=False)
print(f'Loaded : {df_sample.shape[0]:,} rows x {df_sample.shape[1]} cols')
print(f'TANIKODU sample:\n{df_sample["TANIKODU"].dropna().head(10).to_string()}')

# ── Apply severity lookup ──────────────────────────────────────────────────────
# TANIKODU may be pipe-separated (multiple codes); take the max severity
def get_max_severity(tanikodu_val):
    if pd.isna(tanikodu_val):
        return np.nan
    codes = [c.strip() for c in str(tanikodu_val).split('|') if c.strip()]
    severities = [lookup_severity(c) for c in codes]
    valid = [s for s in severities if not (isinstance(s, float) and np.isnan(s))]
    return max(valid) if valid else np.nan

df_sample['SEVERITY_LEVEL'] = df_sample['TANIKODU'].apply(get_max_severity)

# ── Stats ──────────────────────────────────────────────────────────────────────
filled = df_sample['SEVERITY_LEVEL'].notna().sum()
print(f'\nSEVERITY_LEVEL filled : {filled:,} / {len(df_sample):,} ({filled/len(df_sample)*100:.1f}%)')
print(df_sample['SEVERITY_LEVEL'].value_counts().sort_index())

Loaded : 10,000 rows x 851 cols
TANIKODU sample:
0                  L30.9
1                  Z00.0
2                  J20.9
3                    K30
4                    N40
5                  U07.3
6                    N87
7    R50 | K61.0 | L08.8
8                  C50.9
9          K63.9 | K31.9

SEVERITY_LEVEL filled : 9,999 / 10,000 (100.0%)
SEVERITY_LEVEL
1.0    2630
2.0    1050
3.0    2372
4.0    1855
5.0    2092
Name: count, dtype: int64


In [6]:
# ── Save enriched CSV ─────────────────────────────────────────────────────────
out_csv = Path('/home/alperbulbul/ACU/kutsii_workspace/merged_new_v3_sample10000_enriched.csv')
df_sample.to_csv(out_csv, index=False)
print(f'Saved : {out_csv}')
print(f'Shape : {df_sample.shape[0]:,} rows | {df_sample.shape[1]} columns')

Saved : /home/alperbulbul/ACU/kutsii_workspace/merged_new_v3_sample10000_enriched.csv
Shape : 10,000 rows | 852 columns


In [ ]:
# ── Duplicate check on enriched data ──────────────────────────────────────────
total = len(df_sample)
full_dupes = df_sample.duplicated().sum()
ep_dupes   = df_sample.duplicated(subset=['RF_EPISODE2']).sum()
pt_ep_dupes = df_sample.duplicated(subset=['HASTA_ID', 'RF_EPISODE2']).sum()

print(f"Total rows            : {total:,}")
print(f"Full-row duplicates   : {full_dupes:,}")
print(f"Duplicate RF_EPISODE2 : {ep_dupes:,}")
print(f"Duplicate HASTA_ID + RF_EPISODE2 : {pt_ep_dupes:,}")

if full_dupes > 0:
    print("\nSample full-row duplicates:")
    print(df_sample[df_sample.duplicated(keep=False)][['RF_EPISODE2', 'HASTA_ID', 'TANIKODU', 'SEVERITY_LEVEL']].head(10))

if ep_dupes > 0:
    print(f"\nEpisode IDs appearing more than once (top 10):")
    print(df_sample['RF_EPISODE2'].value_counts()[df_sample['RF_EPISODE2'].value_counts() > 1].head(10))